# Laboratorium 13

### 0. Wczytanie danych
Używając metody `load_svmlight_file` z modułu `sklearn` załaduj plik `lab13_files/breast_cancer`, który składa się z 2 elementów: macierzy `X` o wymiarach `683 x 10` oraz wektora `y` o dlugosci `683`. Macierz `X` opisuje badania 683 pacjentow, a wektor `y` ich diagnozy. 

In [1]:
from pathlib import Path

import numpy as np

# Notebook text uses breast_cancer; original starter code uses breast-cancer.
DATA_PATHS = (Path("lab13_files/breast-cancer"), Path("lab13_files/breast_cancer"))
DATA_PATH = next((path for path in DATA_PATHS if path.is_file()), None)
X = y = None

if DATA_PATH is not None:
    try:
        from sklearn.datasets import load_svmlight_file
    except ModuleNotFoundError:
        print("scikit-learn missing; install it to load the supplied dataset.")
    else:
        X, y = load_svmlight_file(DATA_PATH)
        X = X.toarray()
        print(f"Loaded {X.shape[0]} samples and {X.shape[1]} features from {DATA_PATH}.")
else:
    checked_paths = "\n".join(str(path.resolve()) for path in DATA_PATHS)
    print(
        "Dataset missing. Checked:\n"
        f"{checked_paths}\n"
        "Add either file, then run cells from this one onward."
    )

scikit-learn missing; install it to load the supplied dataset.


### 1. Preprocessing
1.1 Korzystajac z `numpy.where` przenumeruj wektor y, tzn zamień wartości 2 i 4 na -1 i +1. 

1.2 Przeskaluj macierz X, tak żeby wartości w każdej kolumnie mieściły się w zakresie [0,1], innymi słowy przeskaluj niezależnie każdą kolumne $X_j$ wg wzoru:


$$
X_j = \frac{X_j-\min(X_j)}{\max(X_j)-\min(X_j)}
$$

In [2]:
if X is not None:
    if not np.all(np.isin(y, [2, 4])):
        raise ValueError("Expected breast-cancer labels 2 and 4.")

    y = np.where(y == 2, -1, 1)

    feature_min = X.min(axis=0)
    feature_range = X.max(axis=0) - feature_min
    safe_range = np.where(feature_range == 0, 1, feature_range)
    X = (X - feature_min) / safe_range

    print("Labels:", np.unique(y))
    print("Feature ranges:", X.min(axis=0), X.max(axis=0))
else:
    print("Preprocessing skipped: dataset path is missing.")

Preprocessing skipped: dataset path is missing.


### 2. Wizualizacja
Narysuj dane używajac biblioteki `matplotlib`. Użyj wymiarów 1 i 2 z macierzy X, a jako kolory użyj wektora y. Pobaw sie parametrami "s" oraz "alpha" metody `plt.scatter` żeby uzyskać czytelną wizualizacje, np. taką jak poniżej.

<img src="lab13_files/d.png">

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt

if X is not None:
    plt.figure(figsize=(7, 5))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=30, alpha=0.7)
    plt.xlabel("feature 1 (scaled)")
    plt.ylabel("feature 2 (scaled)")
    plt.title("Breast-cancer data")
    plt.colorbar(label="class")
    plt.show()
else:
    print("Visualization skipped: dataset path is missing.")

Visualization skipped: dataset path is missing.


### 3. Regresja 

Przeprowadź regresje liniową wg wzoru:
$$
\beta = (X^TX)^{-1}X^Ty
$$

In [4]:
if X is not None:
    # np.linalg.pinv is stable even when X.T @ X is singular.
    beta = np.linalg.pinv(X.T @ X) @ X.T @ y
    print("beta:", beta)
else:
    beta = None
    print("Regression skipped: dataset path is missing.")

Regression skipped: dataset path is missing.


### 4. Wizualizacja Regresji

Zwizualizuj dane po zrzutowaniu na $\beta$ używajac metody `plt.hist`, tzn. narysuj histogramy $\beta X_+$ oraz
$\beta X_-$ gdzie $X_+$ to wiersze $X$ dla których $y=+1$ i analogicznie $X_-$. Postaraj się wygenerować czytelny wykres podobny do tego poniżej.

<img src="lab13_files/lr1.png">

In [5]:
if beta is not None:
    scores = X @ beta
    plt.figure(figsize=(7, 5))
    plt.hist(scores[y == -1], bins=20, alpha=0.65, label="y = -1")
    plt.hist(scores[y == 1], bins=20, alpha=0.65, label="y = +1")
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.xlabel(r"projection $X\beta$")
    plt.ylabel("number of patients")
    plt.title("Class projections onto beta")
    plt.legend()
    plt.show()
else:
    print("Regression visualization skipped: beta is unavailable.")

Regression visualization skipped: beta is unavailable.


### 5. Klasyfikacja

Przeprowadź "klasyfikacje" naszych pacjentów, ci, którzy zostali zrzutowani na wartość ujemną dostają etykiete $-1$, a ci którzy na wartość dodatnia - etykiete $+1$

In [6]:
if beta is not None:
    y_pred = np.where(X @ beta < 0, -1, 1)
    print("Predicted labels:", y_pred)
else:
    y_pred = None
    print("Classification skipped: beta is unavailable.")

Classification skipped: beta is unavailable.


### 6. Ewaluacja

Policz jak dokładny jest ten model, tzn. jak duży jest zbiór
$ \{ (x_i,y_i) : \mathrm{sign}(\beta x_i) = y_i \} $
w stosunku do rozmiaru całego $y$

Powinno udać się uzyskać wynik na poziomie 93-94%

In [7]:
if y_pred is not None:
    accuracy = np.mean(y_pred == y)
    print(f"Accuracy: {accuracy:.2%}")
else:
    accuracy = None
    print("Evaluation skipped: predictions are unavailable.")

Evaluation skipped: predictions are unavailable.


### 7. Regularyzacja 
Dodaj parametr `C` do regresji liniowej
$$
\beta = (X^TX + \tfrac{1}{C}I)^{-1}X^Ty
$$

gdzie $I$ - macierz identyczności. 

Dla jakiej wartości `C` wynik jest najlepszy? Jaki wynik udało Ci sie uzyskać?

In [8]:
if X is not None:
    C_values = np.logspace(-4, 4, 100)
    identity = np.eye(X.shape[1])
    regularized_results = []

    for C in C_values:
        beta_C = np.linalg.solve(X.T @ X + identity / C, X.T @ y)
        prediction_C = np.where(X @ beta_C < 0, -1, 1)
        regularized_results.append((C, np.mean(prediction_C == y)))

    best_C, best_accuracy = max(regularized_results, key=lambda result: result[1])
    print(f"Best C: {best_C:.5g}")
    print(f"Best accuracy: {best_accuracy:.2%}")
else:
    regularized_results = []
    print("Regularization search skipped: dataset path is missing.")

Regularization search skipped: dataset path is missing.


### 8. Gratulacje
Właśnie stworzyłeś/-aś swój pierwszy klasyfikator (statystyczny model predykcji). 

Spróbuj zastosować go do innych danych.

### Zadanie dodatkowe

Zaimplementuj powyższą metode jako klase `LRClassifier` wyposażoną w następujące metody:

* konstruktor przyjmujący dwa parametry: 
    * `C` - stała z równania 
    * `scale` - zmienna boolowska (domyslnie `True`) która określa czy macierz `X` ma byc automatycznie znormalizowana (przeskalowana do `[0,1]`)
* `.fit(X, y)` która spowoduje obliczenie $\beta$, dla `X` - dowolnej macierzy, `y` - wektora etykiet skladajacego sie z dwóch unikalnych wartosci (mogą to byc np. 2 różne stringi)
* `.predict(X)`, która zwróci wektor zawierajacy na `i`tej pozycji etykiete "przewidzianą" przez nasz model (pamiętaj o skalowaniu jesli `scale==True`!)

Przykladowe użycie:

    clf = LRClassifier(C=100)
    Xt = np.array([[1, 0], [1, 1], [1, 2], [1, 3], [1, 4]])
    yt = np.array(['x', 'x', 'y', 'y', 'y'])
    clf.fit(Xt, yt)
    X = np.array([[1, -1], [1, 0.9], [1, 2.1]])
    print(clf.predict(X)) # ['x', 'x', 'y']

In [9]:
class LRClassifier:
    """Binary linear classifier with optional min-max feature scaling."""

    def __init__(self, C=1.0, scale=True):
        if C <= 0:
            raise ValueError("C must be positive.")
        self.C = float(C)
        self.scale = bool(scale)

    def _validate_X(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X must be a two-dimensional matrix.")
        if hasattr(self, "n_features_in_") and X.shape[1] != self.n_features_in_:
            raise ValueError("X has a different number of features than fit data.")
        return X

    def _scale_X(self, X):
        if not self.scale:
            return X

        scaled = X.copy()
        varying = self.feature_range_ != 0
        scaled[:, varying] = (
            (X[:, varying] - self.feature_min_[varying])
            / self.feature_range_[varying]
        )
        # Keep constant columns unchanged; this preserves an intercept column.
        return scaled

    def fit(self, X, y):
        X = self._validate_X(X)
        y = np.asarray(y)
        if y.ndim != 1 or len(y) != len(X):
            raise ValueError("y must be one-dimensional and match X rows.")

        classes = np.unique(y)
        if len(classes) != 2:
            raise ValueError("y must contain exactly two unique labels.")

        self.n_features_in_ = X.shape[1]
        self.feature_min_ = X.min(axis=0)
        self.feature_range_ = X.max(axis=0) - self.feature_min_
        self.classes_ = classes

        X_fit = self._scale_X(X)
        y_signed = np.where(y == self.classes_[0], -1, 1)
        system = X_fit.T @ X_fit + np.eye(self.n_features_in_) / self.C
        self.beta_ = np.linalg.solve(system, X_fit.T @ y_signed)
        return self

    def predict(self, X):
        if not hasattr(self, "beta_"):
            raise ValueError("Call fit before predict.")

        X = self._scale_X(self._validate_X(X))
        return np.where(X @ self.beta_ < 0, self.classes_[0], self.classes_[1])


# Synthetic-data check from task description; independent of external dataset.
Xt = np.array([[1, 0], [1, 1], [1, 2], [1, 3], [1, 4]])
yt = np.array(["x", "x", "y", "y", "y"])
synthetic_predictions = LRClassifier(C=100).fit(Xt, yt).predict(
    [[1, -1], [1, 0.9], [1, 2.1]]
)
assert np.array_equal(synthetic_predictions, ["x", "x", "y"])
print(synthetic_predictions)

['x' 'x' 'y']
